In [ ]:
timestamps_loaded <- read.csv("/data/muscat_data/jaguir26/project1_ucsc_phd/timestamps.csv")
head(timestamps_loaded)
tail(timestamps_loaded)

In [ ]:
# Install if not already installed
if (!require(gdpc)) install.packages("gdpc")
if (!require(readr)) install.packages("readr")
if (!require(doParallel)) install.packages("doParallel")

library(gdpc)
library(readr)
library(doParallel)

data <- read_csv("/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/combined_indices_daily_standardized.csv", show_col_types = FALSE)
selected_indices <- c("Solar Flux", "ONI", "WHWP", "GMT", "AMO", "TSA", "TNA", "SOI")
data_subset <- data[, c("Date", selected_indices)]

data_subset <- na.omit(data_subset)
data_matrix <- as.matrix(data_subset[,-1])
data_standardized <- scale(data_matrix)


# GDPC

In [ ]:
gdpc1 <- gdpc(Z = data_standardized, k = 3, crit = "LOO", niter_max = 100)
print(gdpc1)

residuals1 <- residuals(gdpc1)

gdpc2 <- gdpc(Z = residuals1, k = 3, crit = "LOO", niter_max = 100)
print(gdpc2)


In [ ]:
# Explained variance by each component
cat("Explained variance (GDPC1):", gdpc1$expart, "\n")
cat("Explained variance (GDPC2):", gdpc2$expart, "\n")

# Mean Squared Error (MSE) of reconstruction
cat("Reconstruction MSE (GDPC1):", gdpc1$mse, "\n")
cat("Reconstruction MSE (GDPC2):", gdpc2$mse, "\n")

# Chosen criterion (LOO, AIC, BIC, BNG)
cat("LOO criterion (GDPC1):", gdpc1$crit, "\n")
cat("LOO criterion (GDPC2):", gdpc2$crit, "\n")

# Convergence status
cat("Convergence status (GDPC1):", gdpc1$conv, "\n")
cat("Convergence status (GDPC2):", gdpc2$conv, "\n")


In [ ]:
par(mfrow=c(2,1))
plot(gdpc1$f, type="l", main="GDPC 1", xlab="Time", ylab="Component value", col="blue")
plot(gdpc2$f, type="l", main="GDPC 2", xlab="Time", ylab="Component value", col="red")


In [ ]:
# Loadings for GDPC1
par(mfrow=c(2,3))
for(i in 0:5) plot(gdpc1, which="Loadings", which_load=i, main=paste("GDPC1 - Loadings Lag", i))

# Loadings for GDPC2
par(mfrow=c(2,3))
for(i in 0:5) plot(gdpc2, which="Loadings", which_load=i, main=paste("GDPC2 - Loadings Lag", i))


In [ ]:
# Reconstruction from GDPCs
reconstructed1 <- fitted(gdpc1)
reconstructed2 <- fitted(gdpc2)

# Combined reconstruction using both components
combined_reconstruction <- reconstructed1 + reconstructed2

# Plot reconstruction vs original (example for the first series)
plot(data_standardized[,1], type='l', col='black', main='Original vs Reconstructed Series',
     ylab='Standardized Value', xlab='Time')
lines(combined_reconstruction[,1], col='blue')
legend("topright", legend=c("Original","Reconstructed"), col=c("black","blue"), lty=1)


# PCA

In [ ]:
#!/usr/bin/env Rscript
.libPaths(c("~/R/libs", .libPaths()))
print(.libPaths())
library(dplyr)

# Load required packages
if (!require(readr)) install.packages("readr")
if (!require(ggplot2)) install.packages("ggplot2")
if (!require(dplyr)) install.packages("dplyr")
if (!require(lubridate)) install.packages("lubridate")
if (!require(tidyr)) install.packages("tidyr")

library(readr)
library(ggplot2)
library(dplyr)
library(lubridate)
library(tidyr) 
# -------------------------------
# Step 1: Load and Prepare Data
# -------------------------------
data <- read_csv("/data/muscat_data/jaguir26/project1_ucsc_phd/climate_indices/combined_indices_daily_standardized.csv", show_col_types = FALSE)

selected_indices <- c("Solar Flux", "ONI", "WHWP", "GMT", "TSA", "TNA", "SOI")

# Filter relevant columns
data_subset <- data %>%
  select(Date, all_of(selected_indices)) %>%
  mutate(Date = as.Date(Date)) %>%
  filter(Date >= as.Date("1987-01-01") & Date <= as.Date("2023-01-22")) %>%
  drop_na()

# -------------------------------
# Step 2: Standardize the Data
# -------------------------------
data_matrix <- as.matrix(data_subset[, -1])
data_standardized <- scale(data_matrix)

# -------------------------------
# Step 3: Apply PCA
# -------------------------------
pca_result <- prcomp(data_standardized, center = FALSE, scale. = FALSE)

# -------------------------------
# Step 4: Add PC1 to DataFrame and Plot
# -------------------------------
data_subset$PC1 <- pca_result$x[,2]

# Plot PC1 over time
ggplot(data_subset, aes(x = Date, y = PC1)) +
  geom_line(color = "blue", linewidth = 1) +
  labs(
    title = "First Principal Component of Climate Indices (Standard PCA)",
    subtitle = "From Jan 1, 1987 to Feb 25, 2023",
    x = "Date",
    y = "PC1"
  ) +
  theme_minimal()

In [ ]:
pca <- data_subset[,c('Date','PC1')]
names(pca) <- c('time','Static_PCA')

In [ ]:
write.csv(pca, "/data/muscat_data/jaguir26/project1_ucsc_phd/pca.csv", row.names = FALSE)